In [1]:
!nvidia-smi


Mon Oct 26 15:19:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics tensorrt onnx onnxslim
import ultralytics
import torch
import tensorrt as trt
print(f"TensorRT   : {trt.__version__}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
TensorRT   : 11.4.0.106


In [3]:
from ultralytics import YOLO

PT_PATH = "/kaggle/input/datasets/phnguyencs/detex-pt/best.pt"
model_pt = YOLO(PT_PATH)


import shutil
from pathlib import Path
shutil.copy(PT_PATH, '/kaggle/working/best.pt') 

model_pt = YOLO('/kaggle/working/best.pt')
engine_path = model_pt.export(
    format='engine', 
    device=0,
    half=True,          
    imgsz=640,          
    verbose=False,
)
print(f" Engine saved: {engine_path}")
print(f" Size: {Path(engine_path).stat().st_size / 1e6:.1f} MB")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ Invalid export format='rm engine', updating to format='engine'
Ultralytics 8.4.189 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26m summary (fused): 130 layers, 20,351,765 parameters, 0 gradients, 68.1 GFLOPs

PyTorch: starting from '/kaggle/working/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (42.0 MB)
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 5 packages in 308ms
Prepared 1 package in 3.12s
Installed 1 package in 16ms
 + onnxruntime-gpu==1.31.0

requirements: AutoUpdate success ✅ 4.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting ex

In [4]:
import numpy as np

model_trt = YOLO(engine_path) #lazy loading

dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

print("Warming up (5 runs)...")
for _ in range(5):
    _ = model_trt.predict(dummy, conf=0.35, verbose=False, device=0)
print("Done")

Warming up (5 runs)...
Loading /kaggle/working/best.engine for TensorRT inference...
[10/26/2026-15:25:27] [TRT] [I] Loaded engine size: 181 MiB
[10/26/2026-15:25:27] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +34, now: CPU 0, GPU 214 (MiB)
Done


In [5]:
import time

N_RUNS = 100
CONF   = 0.35
dummy  = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)


model_base = YOLO('/kaggle/working/best.pt')
# Warm-up
for _ in range(3):
    model_base.predict(dummy, conf=CONF, verbose=False, device=0)

t0 = time.perf_counter()
for _ in range(N_RUNS):
    model_base.predict(dummy, conf=CONF, verbose=False, device=0)
torch.cuda.synchronize()
pt_ms = (time.perf_counter() - t0) / N_RUNS * 1000


t0 = time.perf_counter()
for _ in range(N_RUNS):
    model_trt.predict(dummy, conf=CONF, verbose=False, device=0)
torch.cuda.synchronize()
trt_ms = (time.perf_counter() - t0) / N_RUNS * 1000

speedup = pt_ms / trt_ms

print(f"{'':=<50}")
print(f"  PyTorch  FP32 : {pt_ms:6.2f} ms/frame  ({1000/pt_ms:5.1f} FPS)")
print(f"  TensorRT FP16 : {trt_ms:6.2f} ms/frame  ({1000/trt_ms:5.1f} FPS)")
print(f"  Speedup       : {speedup:.2f}x")
print(f"{'':=<50}")

  PyTorch  FP32 :  26.53 ms/frame  ( 37.7 FPS)
  TensorRT FP16 :   7.11 ms/frame  (140.7 FPS)
  Speedup       : 3.73x
